In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
import os

# Set plotting style
plt.style.use('ggplot')
sns.set_palette('muted')

# 1. Load the cleaned dataset
df = pd.read_csv('credit_risk_dataset_cleaned.csv')
print("=== Dataset Overview ===")
print(df.info())
print("\n=== First 5 Rows ===")
print(df.head())

# Check class distribution to identify imbalance
print("\n=== Class Distribution Before Modeling ===")
print(df['default_status'].value_counts(normalize=True))
print(df['default_status'].value_counts())

# 2. Feature Engineering
# Select features and target
features = ['loan_amount', 'loan_term', 'credit_score', 'income', 'debt_to_income']
target = 'default_status'

# Convert categorical variable 'employment_status' to dummy variables
df = pd.get_dummies(df, columns=['employment_status'], drop_first=True)

# Update features to include dummy variables
dummy_features = [col for col in df.columns if col.startswith('employment_status_')]
features.extend(dummy_features)

# Prepare X (features) and y (target)
X = df[features]
y = df[target]

# 3. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Check class distribution in train and test sets
print("\n=== Class Distribution in Training Set ===")
print(pd.Series(y_train).value_counts(normalize=True))
print(pd.Series(y_train).value_counts())
print("\n=== Class Distribution in Test Set ===")
print(pd.Series(y_test).value_counts(normalize=True))
print(pd.Series(y_test).value_counts())

# 4. Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train a Logistic Regression model
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# 6. Evaluate the model
# Predict probabilities
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]  # Probability of default (class 1)

# Save y_test and y_pred_prob for dashboard
pd.DataFrame(y_test, columns=['default_status']).to_csv('y_test.csv', index=False)
pd.DataFrame(y_pred_prob, columns=['pred_prob']).to_csv('y_pred_prob.csv', index=False)
print("Saved y_test.csv and y_pred_prob.csv")

# Adjust threshold to improve recall for Default class
threshold = 0.3  # Lower threshold to increase sensitivity for Default class
y_pred_adjusted = (y_pred_prob >= threshold).astype(int)

# Classification report with adjusted threshold
print("\n=== Classification Report (Adjusted Threshold = 0.3) ===")
print(classification_report(y_test, y_pred_adjusted))

# Confusion matrix with adjusted threshold
print("\n=== Confusion Matrix (Adjusted Threshold = 0.3) ===")
conf_matrix = confusion_matrix(y_test, y_pred_adjusted)
print(conf_matrix)

# ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f"\n=== ROC-AUC Score: {roc_auc:.4f} ===")

# 7. Visualize the results
# Create output directory for plots
if not os.path.exists('plots'):
    os.makedirs('plots')

# (1) Confusion Matrix Heatmap (Improved Visualization)
plt.figure(figsize=(10, 8))  # Increase figure size for clarity
# Create a custom annotation array for dynamic coloring
annot_colors = np.array([['black', 'black'], ['black', 'black']])  # Default to black
for i in range(2):
    for j in range(2):
        if conf_matrix[i, j] > 1000:  # Use white for large values (darker background)
            annot_colors[i, j] = 'white'
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'],
            annot_kws={'size': 12})  # Increase font size
# Apply dynamic colors to annotations
for i in range(2):
    for j in range(2):
        plt.text(j + 0.5, i + 0.5, conf_matrix[i, j],
                 ha='center', va='center', color=annot_colors[i, j], size=12)
plt.title('Confusion Matrix (Threshold = 0.3)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('plots/confusion_matrix_adjusted.png', dpi=300)  # High resolution
plt.close()

# (2) ROC Curve (Improved Visualization with Blue Color)
plt.figure(figsize=(10, 8))
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})', color='blue', linewidth=2)  # Blue color
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess', linewidth=1.5)
plt.grid(True, linestyle='--', alpha=0.7)  # Add grid
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.tick_params(axis='both', labelsize=10)
plt.tight_layout()
plt.savefig('plots/roc_curve.png', dpi=300)  # High resolution
plt.close()

# (3) Predicted Probability Distribution (Improved Visualization)
plt.figure(figsize=(10, 8))
sns.histplot(y_pred_prob, bins=50, kde=True, color='lightblue', stat='count')  # Light blue histogram
# Customize KDE curve
sns.kdeplot(y_pred_prob, color='blue', linewidth=2)  # Thicker KDE curve
# Add threshold line
plt.axvline(x=0.3, color='red', linestyle='--', label='Threshold = 0.3', linewidth=1.5)
plt.title('Distribution of Predicted Default Probabilities', fontsize=14)
plt.xlabel('Predicted Probability of Default', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=12)
plt.tick_params(axis='both', labelsize=10)
plt.tight_layout()
plt.savefig('plots/predicted_probability_distribution.png', dpi=300)  # High resolution
plt.close()

# 8. Save the model coefficients
# Get feature coefficients
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
})
print("\n=== Model Coefficients ===")
print(coef_df)

# Save coefficients to CSV
coef_df.to_csv('model_coefficients.csv', index=False)
print("\nModel coefficients saved as 'model_coefficients.csv'")

=== Dataset Overview ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          20000 non-null  int64  
 1   loan_amount          20000 non-null  float64
 2   loan_term            20000 non-null  int64  
 3   credit_score         20000 non-null  float64
 4   income               20000 non-null  float64
 5   employment_status    20000 non-null  object 
 6   debt_to_income       20000 non-null  float64
 7   default_probability  20000 non-null  float64
 8   default_status       20000 non-null  int64  
dtypes: float64(5), int64(3), object(1)
memory usage: 1.4+ MB
None

=== First 5 Rows ===
   customer_id   loan_amount  loan_term  credit_score        income  \
0            1  57450.712295         12    650.180285  19525.467723   
1            2  47926.035482         24    732.608014  40204.017473   
2            3 